In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# Can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**1. Data Cleaning: Data information and NA removal**

In [ ]:
#Load in Data
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
#Inspect data structure and missing values
print(train.info())
print(train.isnull().sum().sort_values(ascending=False))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
#log-transform skewed features to reduce outliers
train['SalePrice'] = np.log1p(train['SalePrice'])

##AI generated code to drop overly missing columns
threshold = 0.5  # 50%
missing_percent = train.isnull().mean()
cols_to_drop = missing_percent[missing_percent > threshold].index
print(list(cols_to_drop))
train = train.drop(cols_to_drop, axis=1)
test = test.drop(cols_to_drop, axis=1)

# Get percentages for dropped columns
dropped_missing_percent = missing_percent[cols_to_drop].sort_values(ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(dropped_missing_percent.index, dropped_missing_percent.values)
plt.ylabel("Missing Percent")
plt.xlabel("Dropped Variable")
plt.title("Variables Dropped due to High Missingness")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**2. EDA with plots: Histograms and HeatMap**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of SalePrice
plt.figure(figsize=(8,4))
sns.histplot(train['SalePrice'], kde=True)
plt.title('Histogram of SalePrice')
plt.xlabel('SalePrice')
plt.ylabel('Frequency')
plt.show()

# Histogram of log(SalePrice)
plt.figure(figsize=(8,4))
sns.histplot(np.log(train['SalePrice']), kde=True)
plt.title('Histogram of Log(SalePrice)')
plt.xlabel('Log(SalePrice)')
plt.ylabel('Frequency')
plt.show()

numeric_df = train.select_dtypes(include=[np.number])

# Calculate correlation matrix on numeric columns only
corr = numeric_df.corr()

# Find top 10 variables most correlated with SalePrice (excluding SalePrice itself)
top_corr_features = corr['SalePrice'].abs().sort_values(ascending=False).index[1:11]

# Get data for these top features plus SalePrice, dropping rows with NaNs
heatmap_data = numeric_df[top_corr_features.tolist() + ['SalePrice']].dropna()

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data.corr(), annot=True, cmap='coolwarm')
plt.title('Heatmap: Top 10 Numeric Features Correlation with SalePrice')
plt.show()

plt.figure(figsize=(14, 6))
sns.boxplot(data=train, x='Neighborhood', y='SalePrice')
plt.title('Boxplots of Log SalePrice by Neighborhood')
plt.xticks(rotation=45)
plt.show()


**3. Test All Models and Compare RMSE values**

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import numpy as np
import pandas as pd

# Prepare your data (numeric only, fill NaNs for this demo)
X = train.drop(['SalePrice', 'Id'], axis=1).select_dtypes(include=[np.number]).fillna(0)
y = train['SalePrice']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

results = {}

# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_val)
results['Linear Regression'] = np.sqrt(mean_squared_error(y_val, pred_lr))

# 2. Neural Net (simple MLP)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
nn = Sequential([Dense(64, activation='relu', input_dim=X_train.shape[1]), Dense(1)])
nn.compile(optimizer='adam', loss='mse')
nn.fit(X_train_scaled, y_train, epochs=25, verbose=0)
pred_nn = nn.predict(X_val_scaled).flatten()
results['Neural Net'] = np.sqrt(mean_squared_error(y_val, pred_nn))

# 3. Decision Tree
dt = DecisionTreeRegressor()
dt.fit(X_train, y_train)
pred_dt = dt.predict(X_val)
results['Decision Tree'] = np.sqrt(mean_squared_error(y_val, pred_dt))

# 4. Random Forest
rf = RandomForestRegressor(n_estimators=50)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_val)
results['Random Forest'] = np.sqrt(mean_squared_error(y_val, pred_rf))

# 5. Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=50)
gb.fit(X_train, y_train)
pred_gb = gb.predict(X_val)
results['Gradient Boosting'] = np.sqrt(mean_squared_error(y_val, pred_gb))

# 6. XGBoost
xg = xgb.XGBRegressor(n_estimators=50)
xg.fit(X_train, y_train)
pred_xg = xg.predict(X_val)
results['XGBoost'] = np.sqrt(mean_squared_error(y_val, pred_xg))

# 7. LightGBM
lg = lgb.LGBMRegressor(n_estimators=50)
lg.fit(X_train, y_train)
pred_lg = lg.predict(X_val)
results['LightGBM'] = np.sqrt(mean_squared_error(y_val, pred_lg))

# 8. CatBoost
cb = CatBoostRegressor(verbose=0, n_estimators=50)
cb.fit(X_train, y_train)
pred_cb = cb.predict(X_val)
results['CatBoost'] = np.sqrt(mean_squared_error(y_val, pred_cb))

# Create a table of RMSE values
rmse_table = pd.DataFrame(list(results.items()), columns=['Model', 'RMSE'])
print(rmse_table)


**4. Continuing with Gradient Boosting**
- Plotting top 10 variable importances

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Obtain feature importances from your model (assuming 'gb' is your trained GradientBoostingRegressor)
importances = gb.feature_importances_
feature_names = X_train.columns

# Create a DataFrame for easy sorting
feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
top10 = feat_imp_df.sort_values('Importance', ascending=False).head(10)

# Plot bar chart
plt.figure(figsize=(10, 6))
plt.barh(top10['Feature'][::-1], top10['Importance'][::-1])  # reverse to have most important on top
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances (Gradient Boosting)')
plt.tight_layout()
plt.show()


**5. Producing Predictions with Gradient Boosint Model**

In [ ]:
# Prepare test data (make sure preprocessing matches how you handled train data)
X_test = test.drop(['Id'], axis=1).select_dtypes(include=[np.number]).fillna(0)

# Produce predictions
predictions = gb.predict(X_test)
predictions = np.expm1(predictions)


# If your model was trained on the log-transformed SalePrice, invert the transform:
# predictions = np.expm1(predictions)

# Prepare for submission (using sample_submission.csv for format)
submission = pd.DataFrame({'Id': test['Id'], 'SalePrice': predictions})
submission.to_csv('submission.csv', index=False)
print(submission.head())


**Submission file loks good. Let move Forwards**
